# 02 — 데이터 정제 (Clean)

**목적:** `01_audit`에서 확인된 품질 이슈를 규정된 규칙대로 처리한다.  
리텐션·세그먼트 지표 계산은 이 노트북에서 하지 않는다 — 그것은 `03_features`에서 한다.  
**입력:** `data/raw/` (읽기 전용)  
**출력:** `data/processed/events_clean.csv`, `data/processed/profile_clean.csv`

---

## 정제 규칙 요약표

| # | 이슈 | 규모 | 처리 방법 | 사유 |
|---|---|---|---|---|
| R-01 | Event_Type 결측 + 알림_유형 있음 | 3,339건 | Event_Type = `알림수신` 으로 **복원** (알림_유형 보존) | 알림_유형이 복원 단서이므로 NULL 처리 불가 |
| R-02 | Event_Type 결측 + 알림_유형 없음 (R-01 이후) | 23,117건 | 행 **삭제** | 이벤트 유형 식별 불가, 리텐션 산정에 사용 불가 |
| R-03 | 복합키(User_ID·Event_Time·Event_Type) 중복 | 11건 | `keep='first'` 로 **중복 제거** | 동일 이벤트 중복 적재로 판단 |
| R-04 | Event_Time 타입 | 전체 | `datetime64` 변환 | 시계열 연산 필요 |
| R-05 | 로그 수집 장애 구간 이벤트 (2025-03-10 ~ 14) | ~20,586건 | 삭제 대신 `is_outage_period=True` **플래그** | 민감도 분석용 격리 |
| R-06 | Session_ID 결측 (알림 이벤트 등) | 241,502건 | **보정 안 함** | 알림 이벤트는 세션 없음이 정상; 리텐션 분석에 무관 |
| R-07 | 알림_유형 결측 (비알림 이벤트) | 구조적 | **보정 안 함** | 비알림 이벤트에 채워지지 않는 게 정상 |
| R-08 | 가입경로·기기 결측 | 137·121건 | `'Unknown'` 범주 부여 | 해당 유저 제외 시 편향 위험 |
| R-09 | 알림수신동의여부 결측 | 116건 | `True→동의`, `False→미동의`, 결측→`Unknown` | 3분류 명확화 |
| R-10 | 가입일자·변경일자 타입 | 전체 | `datetime64` 변환 (변경일자 결측은 NaT 유지) | 변경일자 결측은 '변경 없음' 의미 |
| R-11 | 장애 구간 가입자 (2025-03-10 ~ 14) | ~341명 | 삭제 대신 `signup_during_outage=True` **플래그** | 포함/제외 민감도 분석용 |

In [3]:
import pandas as pd
import numpy as np

RAW       = '../data/raw'
PROCESSED = '../data/processed'

OUTAGE_START = pd.Timestamp('2025-03-10')
OUTAGE_END   = pd.Timestamp('2025-03-14 23:59:59')

print('라이브러리 로드 완료')

라이브러리 로드 완료


---
## Part 1 — Event_Log 정제

In [4]:
# 원본 로드
events = pd.read_csv(f'{RAW}/Event_Log.csv', encoding='utf-8-sig')

n0 = len(events)
print(f'[원본] 이벤트 행 수: {n0:,}')
print(f'       Event_Type 결측: {events["Event_Type"].isnull().sum():,}')

[원본] 이벤트 행 수: 1,757,262
       Event_Type 결측: 26,456


In [5]:
# R-01 | 복원: Event_Type 결측 & 알림_유형 있음 → '알림수신'
# 반드시 R-02(삭제)보다 먼저 실행해야 한다
mask_restore = events['Event_Type'].isnull() & events['알림_유형'].notna()
n_restore = mask_restore.sum()
events.loc[mask_restore, 'Event_Type'] = '알림수신'

print(f'[R-01] 복원 건수: {n_restore:,}')
print(f'       복원 후 Event_Type 결측: {events["Event_Type"].isnull().sum():,}')

[R-01] 복원 건수: 3,339
       복원 후 Event_Type 결측: 23,117


In [6]:
# R-02 | 삭제: 복원 후에도 Event_Type 결측인 행 → 행 제거
mask_drop = events['Event_Type'].isnull()
n_drop = mask_drop.sum()
events = events[~mask_drop].reset_index(drop=True)

print(f'[R-02] 삭제 건수: {n_drop:,}')
print(f'       삭제 후 행 수: {len(events):,}')
print(f'       Event_Type 결측: {events["Event_Type"].isnull().sum()}')

[R-02] 삭제 건수: 23,117
       삭제 후 행 수: 1,734,145
       Event_Type 결측: 0


In [7]:
# R-03 | 중복 제거: (User_ID, Event_Time, Event_Type) 복합키 keep='first'
DUP_KEY = ['User_ID', 'Event_Time', 'Event_Type']
n_dup = events.duplicated(DUP_KEY).sum()
events = events.drop_duplicates(subset=DUP_KEY, keep='first').reset_index(drop=True)

print(f'[R-03] 중복 제거 건수: {n_dup:,}')
print(f'       제거 후 행 수: {len(events):,}')

[R-03] 중복 제거 건수: 11
       제거 후 행 수: 1,734,134


In [8]:
# R-04 | Event_Time → datetime 변환
events['Event_Time'] = pd.to_datetime(events['Event_Time'])

print(f'[R-04] Event_Time dtype: {events["Event_Time"].dtype}')
print(f'       범위: {events["Event_Time"].min()} ~ {events["Event_Time"].max()}')

[R-04] Event_Time dtype: datetime64[us]
       범위: 2025-01-01 07:00:07 ~ 2025-06-30 22:59:51


In [9]:
# R-05 | 로그 수집 장애 구간 이벤트 플래그 (삭제 아님)
events['is_outage_period'] = events['Event_Time'].between(OUTAGE_START, OUTAGE_END)
n_outage = events['is_outage_period'].sum()

print(f'[R-05] is_outage_period=True 이벤트: {n_outage:,}')
print(f'       전체 행 수 변화 없음: {len(events):,}')

[R-05] is_outage_period=True 이벤트: 20,586
       전체 행 수 변화 없음: 1,734,134


In [10]:
# R-06, R-07 | Session_ID·알림_유형 결측 — 보정 안 함
n_sess_null  = events['Session_ID'].isnull().sum()
n_ntype_null = events['알림_유형'].isnull().sum()
print(f'[R-06] Session_ID 결측 (알림 이벤트 포함, 보정 없음): {n_sess_null:,}')
print(f'[R-07] 알림_유형 결측 (구조적, 보정 없음)           : {n_ntype_null:,}')

# 검증: Event_Type 분포 확인
print()
print('정제 후 Event_Type 분포:')
print(events['Event_Type'].value_counts())

[R-06] Session_ID 결측 (알림 이벤트 포함, 보정 없음): 241,165
[R-07] 알림_유형 결측 (구조적, 보정 없음)           : 1,515,252

정제 후 Event_Type 분포:
Event_Type
앱실행       728650
수면기록      242978
알림수신      197663
운동기록      131267
마음챙김      130344
식단기록      101366
챌린지참여      96828
챌린지_탐색     78100
알림오픈       21219
온보딩_완료      5719
Name: count, dtype: int64


In [19]:
print(events)

          User_ID          Event_Time Event_Type  Session_ID   알림_유형  \
0        U0000001 2025-01-25 07:25:45        앱실행  2858201769     NaN   
1        U0000001 2025-01-25 07:26:15     온보딩_완료  2858201769     NaN   
2        U0000001 2025-01-25 07:26:55     챌린지_탐색  2858201769     NaN   
3        U0000001 2025-01-25 07:27:55      챌린지참여  2858201769     NaN   
4        U0000001 2025-01-25 20:30:00       알림수신         NaN     광고성   
...           ...                 ...        ...         ...     ...   
1734129  U0012500 2025-05-25 22:43:48        앱실행  c93feee9be     NaN   
1734130  U0012500 2025-05-25 22:44:18       수면기록  c93feee9be     NaN   
1734131  U0012500 2025-05-26 10:08:00       알림수신         NaN  챌린지_알림   
1734132  U0012500 2025-05-26 10:41:35        앱실행  591113d3d1     NaN   
1734133  U0012500 2025-05-26 10:57:49        앱실행  76055a2516     NaN   

         is_outage_period  
0                   False  
1                   False  
2                   False  
3                   Fal

---
## Part 2 — User_Profile 정제

In [11]:
# 원본 로드
profile = pd.read_csv(f'{RAW}/User_Profile.csv', encoding='utf-8-sig')

n_profile = len(profile)
print(f'[원본] Profile 행 수: {n_profile:,}  (정제 후에도 동일해야 함)')

[원본] Profile 행 수: 12,500  (정제 후에도 동일해야 함)


In [12]:
# R-08 | 가입경로·기기 결측 → 'Unknown'
n_route_null = profile['가입경로'].isnull().sum()
n_device_null = profile['기기'].isnull().sum()

profile['가입경로'] = profile['가입경로'].fillna('Unknown')
profile['기기']     = profile['기기'].fillna('Unknown')

print(f'[R-08] 가입경로 Unknown 처리: {n_route_null:,}건')
print(f'       기기     Unknown 처리: {n_device_null:,}건')
print(f'       가입경로 분포: {dict(profile["가입경로"].value_counts())}')
print(f'       기기     분포: {dict(profile["기기"].value_counts())}')

[R-08] 가입경로 Unknown 처리: 137건
       기기     Unknown 처리: 121건
       가입경로 분포: {'퍼포먼스광고': np.int64(6852), '오가닉': np.int64(5511), 'Unknown': np.int64(137)}
       기기     분포: {'iOS': np.int64(7175), 'Android': np.int64(5204), 'Unknown': np.int64(121)}


In [13]:
# R-09 | 알림수신동의여부 → 동의/미동의/Unknown 3분류
n_consent_null = profile['알림수신동의여부'].isnull().sum()

profile['알림수신동의여부'] = (
    profile['알림수신동의여부']
    .map({True: '동의', 'True': '동의', False: '미동의', 'False': '미동의'})
    .fillna('Unknown')
)

print(f'[R-09] 동의여부 Unknown 처리: {n_consent_null:,}건')
print(f'       분포: {dict(profile["알림수신동의여부"].value_counts())}')

[R-09] 동의여부 Unknown 처리: 116건
       분포: {'동의': np.int64(7984), '미동의': np.int64(4400), 'Unknown': np.int64(116)}


In [14]:
# R-10 | 가입일자·알림수신동의_변경일자 → datetime (변경일자 결측은 NaT 유지)
profile['가입일자'] = pd.to_datetime(profile['가입일자'])
profile['알림수신동의_변경일자'] = pd.to_datetime(profile['알림수신동의_변경일자'], errors='coerce')

print(f'[R-10] 가입일자 dtype: {profile["가입일자"].dtype}')
print(f'       알림수신동의_변경일자 dtype: {profile["알림수신동의_변경일자"].dtype}')
print(f'       변경일자 NaT (변경 없음): {profile["알림수신동의_변경일자"].isnull().sum():,}건')

[R-10] 가입일자 dtype: datetime64[us]
       알림수신동의_변경일자 dtype: datetime64[us]
       변경일자 NaT (변경 없음): 10,524건


In [15]:
# R-11 | 장애 구간 가입자 플래그 (삭제 아님)
profile['signup_during_outage'] = profile['가입일자'].between(OUTAGE_START, OUTAGE_END)
n_signup_outage = profile['signup_during_outage'].sum()

print(f'[R-11] signup_during_outage=True 유저: {n_signup_outage:,}')
print(f'       Profile 행 수 유지: {len(profile):,}')

[R-11] signup_during_outage=True 유저: 341
       Profile 행 수 유지: 12,500


---
## Part 3 — 최종 검증

In [16]:
# 검증 기준과 비교
checks = [
    ('이벤트 최종 행 수',         len(events),                           1_734_134),
    ('Event_Type 결측',           events['Event_Type'].isnull().sum(),   0),
    ('is_outage_period=True',     int(events['is_outage_period'].sum()),  20_586),
    ('signup_during_outage=True', int(profile['signup_during_outage'].sum()), 341),
    ('Profile 행 수',             len(profile),                          12_500),
]

header = '{:<30} {:>12} {:>12}  {}'.format('항목', '실측', '기준', '판정')
print(header)
print('-' * 62)
all_pass = True
for name, actual, expected in checks:
    ok = '✅' if actual == expected else '❌'
    if actual != expected:
        all_pass = False
    print('{:<30} {:>12,} {:>12,}  {}'.format(name, actual, expected, ok))

print()
print('전체 검증:', '✅ PASS' if all_pass else '❌ FAIL — 위 항목 확인 필요')

항목                                       실측           기준  판정
--------------------------------------------------------------
이벤트 최종 행 수                        1,734,134    1,734,134  ✅
Event_Type 결측                             0            0  ✅
is_outage_period=True                20,586       20,586  ✅
signup_during_outage=True               341          341  ✅
Profile 행 수                          12,500       12,500  ✅

전체 검증: ✅ PASS


---
## Part 4 — 저장

In [17]:
import os
os.makedirs(PROCESSED, exist_ok=True)

events.to_csv(f'{PROCESSED}/events_clean.csv',  index=False, encoding='utf-8-sig')
profile.to_csv(f'{PROCESSED}/profile_clean.csv', index=False, encoding='utf-8-sig')

print('저장 완료')
print(f'  events_clean.csv  : {len(events):,}행 × {events.shape[1]}열')
print(f'  profile_clean.csv : {len(profile):,}행 × {profile.shape[1]}열')

저장 완료
  events_clean.csv  : 1,734,134행 × 6열
  profile_clean.csv : 12,500행 × 7열


---
## 정제 요약표

In [18]:
summary = pd.DataFrame([
    {'구분': 'Event_Log', '처리': 'R-01 복원',      '건수': n_restore,                         '비고': 'Event_Type=알림수신 복원, 알림_유형 보존'},
    {'구분': 'Event_Log', '처리': 'R-02 삭제',      '건수': n_drop,                             '비고': 'Event_Type 식별 불가 (알림_유형도 결측)'},
    {'구분': 'Event_Log', '처리': 'R-03 중복 제거', '건수': n_dup,                              '비고': '복합키(User+Time+Type) keep=first'},
    {'구분': 'Event_Log', '처리': 'R-04 datetime',  '건수': '-',                                '비고': 'Event_Time → datetime64'},
    {'구분': 'Event_Log', '처리': 'R-05 플래그',    '건수': int(events['is_outage_period'].sum()), '비고': 'is_outage_period (2025-03-10~14), 삭제 아님'},
    {'구분': 'Profile',   '처리': 'R-08 Unknown',  '건수': n_route_null + n_device_null,       '비고': '가입경로·기기 결측 → Unknown'},
    {'구분': 'Profile',   '처리': 'R-09 3분류',     '건수': n_consent_null,                     '비고': '알림수신동의 → 동의/미동의/Unknown'},
    {'구분': 'Profile',   '처리': 'R-10 datetime',  '건수': '-',                                '비고': '가입일자·변경일자 → datetime64 (결측=NaT)'},
    {'구분': 'Profile',   '처리': 'R-11 플래그',    '건수': int(profile['signup_during_outage'].sum()), '비고': 'signup_during_outage (2025-03-10~14), 삭제 아님'},
])

print('=== 정제 요약 ===')
print(summary.to_string(index=False))
print()
print(f'이벤트: {n0:,} → {len(events):,}  (감소: {n0 - len(events):,}건)')
print(f'Profile: {n_profile:,} → {len(profile):,}  (행 수 변화 없음)')

=== 정제 요약 ===
       구분            처리    건수                                          비고
Event_Log       R-01 복원  3339                Event_Type=알림수신 복원, 알림_유형 보존
Event_Log       R-02 삭제 23117                Event_Type 식별 불가 (알림_유형도 결측)
Event_Log    R-03 중복 제거    11              복합키(User+Time+Type) keep=first
Event_Log R-04 datetime     -                     Event_Time → datetime64
Event_Log      R-05 플래그 20586     is_outage_period (2025-03-10~14), 삭제 아님
  Profile  R-08 Unknown   258                        가입경로·기기 결측 → Unknown
  Profile      R-09 3분류   116                     알림수신동의 → 동의/미동의/Unknown
  Profile R-10 datetime     -             가입일자·변경일자 → datetime64 (결측=NaT)
  Profile      R-11 플래그   341 signup_during_outage (2025-03-10~14), 삭제 아님

이벤트: 1,757,262 → 1,734,134  (감소: 23,128건)
Profile: 12,500 → 12,500  (행 수 변화 없음)


In [20]:
events_clean = pd.read_csv(f'{PROCESSED}/events_clean.csv', encoding='utf-8-sig')
profile_clean = pd.read_csv(f'{PROCESSED}/profile_clean.csv', encoding='utf-8-sig')

display(events_clean)
display(profile_clean)

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,is_outage_period
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN,False
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN,False
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN,False
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN,False
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성,False
...,...,...,...,...,...,...
1734129,U0012500,2025-05-25 22:43:48,앱실행,c93feee9be,NaN,False
1734130,U0012500,2025-05-25 22:44:18,수면기록,c93feee9be,NaN,False
1734131,U0012500,2025-05-26 10:08:00,알림수신,NaN,챌린지_알림,False
1734132,U0012500,2025-05-26 10:41:35,앱실행,591113d3d1,NaN,False


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,signup_during_outage
0,U0000001,2025-01-25,오가닉,iOS,동의,NaN,False
1,U0000002,2025-05-06,오가닉,iOS,미동의,2025-05-24,False
2,U0000003,2025-05-14,오가닉,iOS,미동의,NaN,False
3,U0000004,2025-02-23,퍼포먼스광고,Android,동의,NaN,False
4,U0000005,2025-02-18,퍼포먼스광고,Android,동의,NaN,False
...,...,...,...,...,...,...,...
12495,U0012496,2025-04-24,오가닉,Android,미동의,NaN,False
12496,U0012497,2025-01-28,오가닉,Android,미동의,NaN,False
12497,U0012498,2025-02-05,오가닉,iOS,미동의,NaN,False
12498,U0012499,2025-05-07,오가닉,Android,동의,NaN,False
